<a href="https://colab.research.google.com/github/swarnapatro/AI_KickStart/blob/main/langchain_business_usecase_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Business Use Case: E-commerce Customer Support Assistant

**Companion notebook to the "LangChain Basics" reference deck.**

This notebook builds ONE small, realistic business application end-to-end, using every
concept from the slide deck along the way:

| Deck section | Used here for |
|---|---|
| Embeddings & Ingestion | Turning the store's policy/FAQ text into a searchable knowledge base |
| Tools | A retrieval tool (search policies) and an order-lookup tool (mock database) |
| Agents | One `create_agent()` that decides which tool(s) a question needs |
| Memory | A checkpointer so the assistant remembers the conversation |

### The business scenario

You run the support desk for a small online store, **Northwind Gadgets**. Customers ask two
kinds of questions:

1. Policy/FAQ questions -- *"What's your return window?"* -> answered by searching store documents (RAG).
2. Order questions -- *"Where is order A101?"* -> answered by looking up a live order record (a tool).

Often a single question needs **both** (*"Is my order still eligible for a return?"*), and a
good support chat should **remember earlier turns** so customers don't repeat themselves.

We'll build exactly this, in under 40 lines of "real" code, split into small steps.

> **No coding background?** Read the markdown cells for the concept, skim the code, and focus
> on the printed output at the bottom of each cell -- that's the actual behavior in plain English.

> **This version runs entirely on free, open-source models -- no OpenAI key or paid quota
> needed.** Embeddings run locally on Colab's CPU/GPU for free. The agent's "brain" is an
> open-weight model (`openai/gpt-oss-120b`, Apache-2.0 licensed) served for free through
> Hugging Face's Inference Providers router -- you only need a free Hugging Face account.


## Setup

This notebook needs a free **Hugging Face** account and access token (no credit card, no
paid tier required):

1. Create a free account at [huggingface.co/join](https://huggingface.co/join) if you don't have one.
2. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) -> **Create new token** -> type **"Read"** is enough.
3. Copy the token and paste it when the cell below prompts you (it is only kept for this Colab session, never saved to disk).

We use two free pieces of infrastructure:
- **Local embeddings** (`sentence-transformers/all-MiniLM-L6-v2`) -- downloaded once and run directly on the Colab machine, completely free, no token needed for this part.
- **Hugging Face's Inference Providers router** -- a free, OpenAI-compatible API in front of open-weight models. We point LangChain's `ChatOpenAI` class at it instead of at OpenAI itself, so the rest of the LangChain code (tools, agent, memory) is unchanged from a "normal" OpenAI setup -- only these two cells differ.


In [ ]:
# One-time install (safe to re-run)
%pip install -q langchain langchain-openai langchain-core langchain-text-splitters langgraph langchain-huggingface sentence-transformers


In [ ]:
import os
import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your free Hugging Face access token: ")


Enter your free Hugging Face access token: ··········


## Step 1 -- Ingestion: turn store documents into text chunks

*(Deck section: Embeddings & Ingestion -- the "Load -> Split" part of the pipeline.)*

In a real system these would come from PDFs or a help-center export. Here they're plain
Python strings so the example stays runnable with zero external files.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

store_documents = [
    """Return Policy: Items can be returned within 30 days of delivery for a full refund,
    provided they are unused and in original packaging. Sale items are final and cannot
    be returned. Refunds are issued to the original payment method within 5-7 business days.""",

    """Shipping Policy: Standard shipping takes 3-5 business days within the country.
    Express shipping (2 business days) is available at checkout for an extra fee.
    We do not currently ship internationally.""",

    """Warranty Policy: All electronics carry a 1-year manufacturer warranty covering
    defects, not accidental damage. Contact support with your order ID to start a
    warranty claim.""",

    """Cancellation Policy: Orders can be cancelled for free within 1 hour of placing
    them, as long as they have not shipped. After that, treat it as a return once received.""",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.create_documents(store_documents)

print(f"{len(store_documents)} documents split into {len(chunks)} chunks")
print("---\nExample chunk:\n", chunks[0].page_content)


4 documents split into 4 chunks
---
Example chunk:
 Return Policy: Items can be returned within 30 days of delivery for a full refund,
    provided they are unused and in original packaging. Sale items are final and cannot
    be returned. Refunds are issued to the original payment method within 5-7 business days.


## Step 2 -- Embeddings & Vector Store: make the chunks searchable

*(Deck section: Embeddings & Ingestion -- the "Embed -> Store" part of the pipeline.)*

Each chunk is converted into a vector (a list of numbers capturing its meaning), then stored
so we can later ask "which chunks are closest in meaning to this question?"

We use `HuggingFaceEmbeddings` with a small, free, open-source embedding model
(`all-MiniLM-L6-v2`, ~90MB). It downloads once and then runs locally -- no API key, no
cost, no rate limit, and it's the same `embed_query` / `embed_documents` interface used
in the deck, just pointed at a different (free) model.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Quick sanity check -- does retrieval find the right chunk for a question?
preview = retriever.invoke("How long do I have to return something?")
print(preview[0].page_content)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Return Policy: Items can be returned within 30 days of delivery for a full refund,
    provided they are unused and in original packaging. Sale items are final and cannot
    be returned. Refunds are issued to the original payment method within 5-7 business days.


## Step 3 -- Tools: give the agent hands

*(Deck section: Tools.)*

Two tools:

1. **`search_store_policies`** -- wraps the retriever above, so the agent can search documents.
2. **`get_order_status`** -- a mock order database lookup (in real life: a call to your order API/DB).

Notice both are just plain Python functions with a docstring and type hints -- that docstring
is what the agent reads to decide when to use each one.


In [ ]:
from langchain.tools import tool

@tool
def search_store_policies(question: str) -> str:
    """Search Northwind Gadgets' policy and FAQ documents (returns, shipping,
    warranty, cancellation) for the answer to a customer question."""
    results = retriever.invoke(question)
    return "\n---\n".join(r.page_content for r in results)


# A tiny mock "database" of orders for this demo
_ORDER_DB = {
    "A101": {"status": "Shipped", "delivered_days_ago": 5, "item": "Wireless Mouse"},
    "A102": {"status": "Processing", "delivered_days_ago": None, "item": "Mechanical Keyboard"},
    "A103": {"status": "Delivered", "delivered_days_ago": 40, "item": "USB-C Hub"},
}

@tool
def get_order_status(order_id: str) -> str:
    """Look up the current status of a customer's order by order ID, e.g. 'A101'.
    Returns the shipping status, days since delivery (if delivered), and item name."""
    order = _ORDER_DB.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return str(order)

tools = [search_store_policies, get_order_status]
print("Tools ready:", [t.name for t in tools])


Tools ready: ['search_store_policies', 'get_order_status']


## Step 4 -- Agent with Memory: put it all together

*(Deck sections: Agents + Memory.)*

`create_agent()` combines the model, the two tools, and a `checkpointer` for memory. Every
call using the same `thread_id` continues the same conversation -- the agent automatically
sees earlier turns.

**The open-source model:** instead of connecting to OpenAI, we point `ChatOpenAI` at
Hugging Face's **Inference Providers router** (`https://router.huggingface.co/v1`) -- a
free, OpenAI-compatible API in front of open-weight models. We use `openai/gpt-oss-120b`,
an open-source (Apache-2.0), tool-calling-capable model. Because the router speaks the
same protocol as OpenAI, `create_agent()` and every tool-calling mechanic from the deck
work completely unchanged -- only the connection details differ, right here in one place.

> **Hitting a rate limit or a "model too busy" message?** The free tier can be busy at
> peak times. Swap `model="openai/gpt-oss-120b"` for `model="Qwen/Qwen2.5-72B-Instruct"`
> or `model="meta-llama/Llama-3.3-70B-Instruct"` -- both are also free, open-source, and
> tool-calling capable through the same router, no other code changes needed.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
    model="openai/gpt-oss-120b",   # free, open-source, tool-calling capable
    temperature=0,
)

SYSTEM_PROMPT = (
    "You are a helpful, concise customer support assistant for Northwind Gadgets. "
    "Use the search_store_policies tool for policy/FAQ questions, and get_order_status "
    "for questions about a specific order. Combine both when a question needs it."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

print("Agent ready.")


Agent ready.


## Step 5 -- Try it: a multi-turn support conversation

*(This is where Ingestion, Embeddings, Tools, Agents, and Memory all work together.)*

We'll simulate one customer, one `thread_id`, asking three questions in a row:

1. An order-status question (needs the `get_order_status` tool).
2. A follow-up that only makes sense with memory of question 1 (needs both tools).
3. A pure policy question (needs only `search_store_policies`).


In [ ]:
config = {"configurable": {"thread_id": "customer-demo-1"}}

def ask(question: str):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config,
    )
    answer = result["messages"][-1].content
    print(f"Customer: {question}\nAssistant: {answer}\n")
    return result

_ = ask("Hi, where is my order A101?")
_ = ask("If I don't like it, can I still return it?")   # relies on memory of "A101"
_ = ask("Do you ship internationally?")                  # pure policy question


Customer: Hi, where is my order A101?
Assistant: Your order **A101** (Wireless Mouse) was shipped and was delivered **5 days ago**. Let us know if you need anything else!

Customer: If I don't like it, can I still return it?
Assistant: Yes—you can return the Wireless Mouse.  

**Northwind Gadgets Return Policy**  
- Returns are accepted within **30 days of delivery**.  
- The item must be **unused** and in its **original packaging**.  
- Refunds are issued to the original payment method within 5‑7 business days.  

Since your order was delivered **5 days ago**, you’re well within the return window. Just pack the mouse securely, include the original receipt or order number (A101), and send it back using the return label provided in your shipment or request one from our support team. Let us know if you need a return label or any further assistance!

Customer: Do you ship internationally?
Assistant: We currently **do not ship internationally**. All of our shipping options—standard (3‑5 bu

## Bonus: proving memory actually matters

Ask the *same* follow-up question, but on a **brand-new `thread_id`** -- with no shared
history, the agent can't know which order "it" refers to.


In [ ]:
fresh_config = {"configurable": {"thread_id": "customer-demo-2"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "If I don't like it, can I still return it?"}]},
    fresh_config,
)
print(result["messages"][-1].content)


Yes—you can return an item you don’t like as long as you do it within 30 days of delivery. The product must be unused and in its original packaging. Refunds are issued to your original payment method within 5‑7 business days. (Note: sale items are final and cannot be returned.)


## Recap: how this maps back to the deck

- **Ingestion** -- `store_documents` -> `RecursiveCharacterTextSplitter` -> `chunks`.
- **Embeddings** -- `HuggingFaceEmbeddings` (free, local, open-source) turned each chunk into a vector.
- **Vector store & retriever** -- `InMemoryVectorStore` + `as_retriever()` found relevant chunks by meaning.
- **Tools** -- `search_store_policies` (wraps the retriever) and `get_order_status` (mock DB) -- both plain functions with docstrings.
- **Agent** -- `create_agent()` decided, per question, which tool(s) to call, powered by a free, open-source model (`openai/gpt-oss-120b`) via Hugging Face's Inference Providers router.
- **Memory** -- the `checkpointer` + `thread_id` let the second question resolve "it" correctly.

### Why this still matches the deck exactly
The deck's code examples use OpenAI directly for simplicity, since that's the most common
starting point. This notebook swaps only the **connection details** for the embedding
model and the chat model -- every LangChain concept (tools, `create_agent`, checkpointer,
`thread_id`, retriever) is the identical code and API you saw in the slides. This is the
"swappable component" idea from the deck's architecture section in action: change the
model, keep the app.

### Ideas to extend this yourself
- Swap `InMemoryVectorStore` for a persistent one (Chroma, pgvector, Pinecone) so the knowledge base survives a restart.
- Swap `InMemorySaver()` for a persistent checkpointer so conversations survive a restart.
- Add a third tool (e.g. "create a return label") and require human approval before it runs -- LangGraph supports this out of the box.
- Load `store_documents` from real files with a document loader instead of hard-coded strings.
- If you later get access to a paid provider (OpenAI, Anthropic, etc.), only the `llm = ChatOpenAI(...)` cell needs to change -- swap it for `init_chat_model("gpt-4o-mini", model_provider="openai")` and everything else still works.
